# Lesson 2.5a — Part I：工程化 PDF 解析流水线

**目标**：判类型 → 提取 → 清洗 → 按 token 切 chunk → 输出 JSONL

下一步 → [05b_part2_advanced_parsing.ipynb](05b_part2_advanced_parsing.ipynb)（Marker / LlamaParse，选修）

---


## 完整流水线（Part I）

见下方 Step 1–7。

```mermaid
flowchart LR
    A[PDF] --> B[逐页遍历<br/>fitz.open]
    B --> C[extract_page<br/>判数字/扫描/空页]
    C -->|数字| D[pdfplumber<br/>文本 + 表格 MD]
    C -->|pdfplumber 失败| E[PyMuPDF 回退]
    C -->|扫描| F[OCR<br/>pdf2image + Tesseract]
    D --> G[clean_text<br/>连字符/空白]
    E --> G
    F --> G
    G --> H[split_into_chunks<br/>tiktoken + overlap]
    H --> I[Chunk<br/>source_type=text/ocr]
    D --> J[Chunk<br/>source_type=table]
    I --> K[process_pdf yield]
    J --> K
    K --> L[JSONL 落盘<br/>out/chunks/pdf_stem/*.jsonl]
    K --> M[all_chunks 内存<br/>供 Part III]

    style C fill:#546E7A,color:#fff
    style D fill:#1565C0,color:#fff
    style E fill:#1976D2,color:#fff
    style F fill:#6A1B9A,color:#fff
    style J fill:#E65100,color:#fff
    style L fill:#2E7D32,color:#fff
    style M fill:#1B5E20,color:#fff
```

## 0. 安装依赖


In [48]:
%pip install PyMuPDF pdfplumber tiktoken pandas -q


Note: you may need to restart the kernel to use updated packages.


In [49]:
import json
import os
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterator

import fitz
import pdfplumber
import tiktoken

from dotenv import load_dotenv

load_dotenv(override=True)


def _resolve_here() -> Path:
    """定位 05_PDF_process 目录（兼容 kernel cwd 为项目根或本目录）。"""
    for base in (Path.cwd(), Path.cwd() / "02-RAG" / "05_PDF_process"):
        if (base / "doc" / "attention_is_all_you_need.pdf").exists():
            return base.resolve()
    return Path.cwd().resolve()


HERE = _resolve_here()
PDF_PATH = HERE / "doc" / "attention_is_all_you_need.pdf"
print(f"工作目录: {HERE}")
print(f"当前 PDF: {PDF_PATH}")

工作目录: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process
当前 PDF: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process\doc\attention_is_all_you_need.pdf


## 1. 配置参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| CHUNK_SIZE_TOKENS | 400 | chunk token 上限 |
| CHUNK_OVERLAP_TOKENS | 50 | overlap |
| TEXT_LEN_THRESHOLD | 50 | 扫描页判定阈值 |

In [50]:
CHUNK_SIZE_TOKENS = 400
CHUNK_OVERLAP_TOKENS = 50
TEXT_LEN_THRESHOLD = 50
ENCODER = tiktoken.get_encoding("cl100k_base")

In [51]:
@dataclass
class Chunk:
    file: str
    page: int
    chunk_id: int
    text: str
    token_count: int
    source_type: str  # text / table / ocr / marker / llamaparse

## 2. Step 1–4 — 按页提取（数字页 pdfplumber，失败回退 PyMuPDF）

In [52]:
def _table_to_markdown(table: list[list[str]]) -> str:
    if not table or len(table) < 2:
        return ""
    header = "| " + " | ".join(str(c or "") for c in table[0]) + " |"
    sep = "| " + " | ".join("---" for _ in table[0]) + " |"
    rows = ["| " + " | ".join(str(c or "") for c in row) + " |" for row in table[1:]]
    return "\n".join([header, sep, *rows])


def _extract_digital(pdf_path: Path, page_num: int) -> tuple[str, list[str]]:
    """数字页：优先 pdfplumber；失败则 PyMuPDF 回退"""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            page = pdf.pages[page_num]
            text = page.extract_text() or ""
            tables = []
            for table in page.extract_tables() or []:
                if table:
                    tables.append(_table_to_markdown(table))
            return text, tables
    except Exception as e:
        print(f"  ⚠️ pdfplumber 第 {page_num+1} 页失败 ({e})，回退 PyMuPDF")
        with fitz.open(pdf_path) as doc:
            return doc[page_num].get_text(), []


def _extract_scanned(pdf_path: Path, page_num: int) -> str:
    try:
        from pdf2image import convert_from_path
        import pytesseract
    except ImportError:
        return "[扫描页，OCR 依赖未安装]"
    images = convert_from_path(pdf_path, dpi=300, first_page=page_num + 1, last_page=page_num + 1)
    return pytesseract.image_to_string(images[0], lang="chi_sim+eng") if images else ""

In [53]:
def extract_page(pdf_path: Path, page_num: int) -> dict:
    with fitz.open(pdf_path) as doc:
        page = doc[page_num]
        raw_text = page.get_text().strip()
        images = page.get_images(full=True)
    if len(raw_text) >= TEXT_LEN_THRESHOLD:
        text, tables = _extract_digital(pdf_path, page_num)
        return {"type": "digital", "text": text, "tables": tables}
    if images:
        return {"type": "scanned", "text": _extract_scanned(pdf_path, page_num), "tables": []}
    return {"type": "empty", "text": "", "tables": []}

In [54]:
page0 = extract_page(PDF_PATH, 0)
print(f"第 1 页: {page0['type']}, 文本 {len(page0['text'])} 字符, 表格 {len(page0['tables'])}")
print(page0["text"][:300])

第 1 页: digital, 文本 2580 字符, 表格 0
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog


## 3. Step 5 — 文本清洗

In [55]:
def clean_text(text: str) -> str:
    text = re.sub(r"-\n(\w)", r"\1", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

## 4. Step 6 — 按 token 切 chunk

用 `tiktoken` 而非按字符切（见 02 notebook 讲义）。

In [56]:
def split_into_chunks(text: str) -> list[str]:
    if not text.strip():
        return []
    tokens = ENCODER.encode(text)
    chunks, start = [], 0
    while start < len(tokens):
        end = start + CHUNK_SIZE_TOKENS
        chunks.append(ENCODER.decode(tokens[start:end]))
        start = end - CHUNK_OVERLAP_TOKENS
    return chunks

## 5. Step 7 — 流水线 → JSONL

In [57]:
def process_pdf(pdf_path: Path) -> Iterator[Chunk]:
    with fitz.open(pdf_path) as doc:
        n_pages = len(doc)
    chunk_id = 0
    for page_num in range(n_pages):
        result = extract_page(pdf_path, page_num)
        cleaned = clean_text(result["text"])
        for piece in split_into_chunks(cleaned):
            yield Chunk(pdf_path.name, page_num + 1, chunk_id, piece,
                        len(ENCODER.encode(piece)),
                        "ocr" if result["type"] == "scanned" else "text")
            chunk_id += 1
        for table_md in result["tables"]:
            yield Chunk(pdf_path.name, page_num + 1, chunk_id, table_md,
                        len(ENCODER.encode(table_md)), "table")
            chunk_id += 1

In [61]:
out_path = HERE / "out" /"chunks"/ f"{PDF_PATH.stem}_chunks.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)

all_chunks: list[Chunk] = []
with out_path.open("w", encoding="utf-8") as f:
    for chunk in process_pdf(PDF_PATH):
        all_chunks.append(chunk)
        f.write(json.dumps(asdict(chunk), ensure_ascii=False) + "\n")
print(f"✅ Part I 完成: {len(all_chunks)} chunks → {out_path}")

✅ Part I 完成: 46 chunks → D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process\out\chunks\attention_is_all_you_need_chunks.jsonl


In [59]:
for obj in all_chunks[:3]:
    print(f"[p{obj.page} #{obj.chunk_id} {obj.source_type}] {obj.token_count} tok | {obj.text[:80]}...")

[p1 #0 text] 400 tok | Providedproperattributionisprovided,Googleherebygrantspermissionto reproducethet...
[p1 #1 text] 331 tok | after trainingfor3.5daysoneightGPUs,asmallfractionofthetrainingcostsofthe bestmo...
[p2 #2 text] 400 tok | 1 Introduction Recurrentneuralnetworks,longshort-termmemory[13]andgatedrecurrent...


---

✅ Part I 产出：`out/chunks/{pdf_stem}/{pdf_stem}_chunks.jsonl`

Part III 会从该文件加载 chunk，无需在同一 kernel 保留 `all_chunks`。
